# 03. Seasonality and holidays

Seasonality is the pattern that repeats on a calendar: the week, the year, the hour of the day. Holidays and events are the special days on top of it. This notebook shows what Prophet fits by default, how to tune it, and how to add patterns of your own.

- Built-in seasonalities
- Fourier order: how wiggly a pattern may be
- Switching seasonalities on and off, and daily seasonality
- A custom seasonality
- Conditional seasonality: the NFL season on and off
- Built-in country holidays
- Custom holidays and events: windows and `holidays_prior_scale`
- Additive or multiplicative seasonality

Previous: [02. Trend](02_trend.ipynb) | Next: [04. Regressors and weather](04_regressors_and_weather.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from prophet import Prophet, forecast
from prophet.make_holidays import make_holidays_df
from prophet.plot import plot_forecast_component, plot_seasonality, plot_yearly

from helpers import load_example, quiet_logs

quiet_logs()

## 1. Built-in seasonalities

Prophet turns on a seasonality when the data can support it: yearly for at least two years of history, weekly for at least two weeks of data that is finer than weekly, and daily for data finer than daily. Peyton Manning's daily page views get yearly and weekly patterns. The fitted settings sit in `m.seasonalities`. We pass `freq="D"` because the series has some missing days.

In [ ]:
df = load_example("peyton_manning")
result = forecast(df, horizon=365, freq="D")
m, fcst = result.model, result.predictions
pd.DataFrame(m.seasonalities).T[["period", "fourier_order", "prior_scale", "mode"]]

In [ ]:
fig = result.plot_components()

Takeaway: page views peak on Mondays and in January and fall in the summer, which follows the NFL calendar.

## 2. Fourier order: how wiggly a pattern may be

Each seasonality is a sum of sine and cosine terms. The Fourier order is the number of terms, and each one adds two parameters. The defaults are 10 for yearly, 3 for weekly and 4 for daily. Pass an integer instead of `"auto"` to change it. A low order gives a smooth curve and a high one can follow sharp features, and noise.

In [ ]:
orders = [3, 10, 40]
by_order = {n: forecast(df, 365, freq="D", yearly_seasonality=n) for n in orders}

fig, axes = plt.subplots(len(orders), 1, figsize=(10, 10), layout="constrained")
for ax, (n, r) in zip(axes, by_order.items()):
    plot_yearly(r.model, ax=ax, uncertainty=False, title=f"yearly_seasonality = {n}")

In [ ]:
def in_sample_rmse(r) -> float:
    fitted = r.predictions.merge(r.history, on="ds")
    return float(np.sqrt(((fitted["yhat"] - fitted["y"]) ** 2).mean()))


pd.Series({n: in_sample_rmse(r) for n, r in by_order.items()}, name="in_sample_rmse").rename_axis("fourier_order").round(3).to_frame()

Takeaway: the in-sample error falls as the order rises, because the model can bend to every year's quirks. That is not evidence of a better forecast; the order-40 curve is chasing noise. Choose the order by cross-validation (notebook 06).

## 3. Switching seasonalities on and off, and daily seasonality

`yearly_seasonality`, `weekly_seasonality` and `daily_seasonality` each take `"auto"`, `True`, `False` or a Fourier order. Daily seasonality needs sub-daily data, so here we use temperature readings from Yosemite, taken every five minutes and averaged to hourly. Temperature has no weekly rhythm, so weekly is switched off. Yearly is off automatically because there are only about two months of data.

In [ ]:
temps = load_example("yosemite_temps").set_index("ds")["y"].resample("h").mean().reset_index()
temps.head()

In [ ]:
temp_result = forecast(temps, horizon="3D", weekly_seasonality=False)
print("Seasonalities fitted:", list(temp_result.model.seasonalities))
fig = temp_result.plot_components()

Takeaway: the daily panel peaks at about 20:30 and bottoms out around midday on the data's clock. For a mountain valley that is only plausible if the timestamps are UTC: 20:30 UTC is early afternoon in California in summer. I have not confirmed the time zone of the source, and it is worth checking before you read the clock times off a daily pattern.

## 4. A custom seasonality

If a cycle is not weekly, yearly or daily, add it with `add_seasonality(name, period, fourier_order)`, where the period is in days. `forecast()` has no argument for this, so we use `Prophet` and wrap the fit-and-predict steps in a small helper. As an example, add a monthly cycle of about 30.5 days to Peyton Manning's page views.

In [ ]:
def fit_and_predict(m: Prophet, df: pd.DataFrame, periods: int = 365, freq: str = "D", add_columns=None) -> pd.DataFrame:
    """Fit ``m`` on ``df`` and predict ``periods`` steps past its end.

    ``add_columns`` is an optional function that adds extra columns, such as a
    seasonality condition, to both the training and the future frame.
    """
    add = add_columns or (lambda frame: frame)
    m.fit(add(df))
    return m.predict(add(m.make_future_dataframe(periods=periods, freq=freq)))


m_monthly = Prophet()
m_monthly.add_seasonality(name="monthly", period=30.5, fourier_order=5)
f_monthly = fit_and_predict(m_monthly, df)
_ = plot_seasonality(m_monthly, "monthly")

In [ ]:
f_monthly[["yearly", "weekly", "monthly"]].agg(lambda s: s.max() - s.min()).rename("swing, peak to trough").round(2).to_frame()

Takeaway: the monthly pattern is small next to the yearly and weekly ones, so for this series it adds parameters without adding much. Add a custom seasonality when you can name the cycle, then check that it earns its place.

## 5. Conditional seasonality: the NFL season on and off

A seasonal pattern can depend on a condition. Peyton Manning's weekly pattern should be strong during the NFL season, September to January, and weak outside it. Replace the built-in weekly seasonality with two, each tied to a boolean column through `condition_name`. The columns must exist in the future frame too, which is why `forecast()` cannot do this and the helper from the last section is reused.

In [ ]:
def is_nfl_season(ds: pd.Series) -> pd.Series:
    return (ds.dt.month > 8) | (ds.dt.month < 2)


def add_season_flags(frame: pd.DataFrame) -> pd.DataFrame:
    on = is_nfl_season(frame["ds"])
    return frame.assign(on_season=on, off_season=~on)


m_cond = Prophet(weekly_seasonality=False)
m_cond.add_seasonality(name="weekly_on_season", period=7, fourier_order=3, condition_name="on_season")
m_cond.add_seasonality(name="weekly_off_season", period=7, fourier_order=3, condition_name="off_season")
f_cond = fit_and_predict(m_cond, df, add_columns=add_season_flags)
f_cond[["ds", "weekly_on_season", "weekly_off_season"]].tail()

In [ ]:
fig = m_cond.plot_components(f_cond)

In [ ]:
f_cond[["weekly_on_season", "weekly_off_season"]].agg(lambda s: s.max() - s.min()).rename("swing, peak to trough").round(2).to_frame()

Takeaway: the weekly swing during the season is about four times the off-season one, and the big jump is on Monday, after Sunday games. The panels use different vertical scales, so compare the swings in the table.

## 6. Built-in country holidays

`country_holidays="US"` adds that country's public holidays as events, computed by the `holidays` package for any year in the history or the horizon. One country at a time is supported. Use a two-letter code such as `"GB"` or a name like `"UnitedStates"`.

In [ ]:
holiday_result = forecast(df, 365, freq="D", country_holidays="US")
hm, hf = holiday_result.model, holiday_result.predictions
names = list(hm.train_holiday_names)
print(len(names), "holidays in the model")

hf[names].where(lambda x: x != 0).mean().sort_values().rename("effect on log views").round(2).to_frame()

In [ ]:
_ = plot_forecast_component(hm, hf, "holidays")

Takeaway: most holidays pull page views down, but each effect is estimated from only about eight occurrences and can pick up something else. Martin Luther King Jr. Day is positive because it falls in late January, in playoff season, which the next section models explicitly. The "observed" entries exist only when a holiday fell on a weekend, so they rest on very few days and are the least reliable.

For a state or province, build a holiday frame with `make_holidays_df` and pass it as `holidays=`. It has the same shape as the custom events in the next section.

In [ ]:
nsw = make_holidays_df(year_list=list(range(2019, 2029)), country="AU", province="NSW")
nsw.head()

## 7. Custom holidays and events: windows and `holidays_prior_scale`

Events that are not public holidays go in a frame with `holiday` and `ds`, one row per occurrence, covering the past and the future. `lower_window` and `upper_window` stretch an event over the days around it. For Peyton Manning, the playoff games and Super Bowls drive large spikes. The dates below are as listed in the upstream Prophet documentation and are used for illustration. Super Bowl dates also appear as playoff games, so the Super Bowl effect is an extra on top of the playoff effect. Two of the dates, 24 January and 7 February 2016, fall after the data ends, so the forecast includes them.

In [ ]:
def events(name: str, dates: list[str], lower_window: int = 0, upper_window: int = 1) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "holiday": name,
            "ds": pd.to_datetime(dates),
            "lower_window": lower_window,
            "upper_window": upper_window,
        }
    )


playoff_dates = [
    "2008-01-13", "2009-01-03", "2010-01-16", "2010-01-24", "2010-02-07", "2011-01-08", "2013-01-12",
    "2014-01-12", "2014-01-19", "2014-02-02", "2015-01-11", "2016-01-17", "2016-01-24", "2016-02-07",
]
superbowl_dates = ["2010-02-07", "2014-02-02", "2016-02-07"]


def playoffs_and_superbowls(lower_window: int = 0, upper_window: int = 1) -> pd.DataFrame:
    return pd.concat(
        [
            events("playoff", playoff_dates, lower_window, upper_window),
            events("superbowl", superbowl_dates, lower_window, upper_window),
        ],
        ignore_index=True,
    )


holidays = playoffs_and_superbowls()
holidays.tail()

In [ ]:
event_result = forecast(df, 365, freq="D", holidays=holidays)
em, ef = event_result.model, event_result.predictions
ef.loc[(ef["playoff"] + ef["superbowl"]).abs() > 0, ["ds", "playoff", "superbowl"]].tail(6)

In [ ]:
_ = plot_forecast_component(em, ef, "holidays")

Takeaway: a playoff game lifts that day and the day after, and the Super Bowl adds a further bump. The effects are in log views, so an effect of 1.2 to 1.9 means roughly three to seven times the usual page views.

**Windows.** With the default window `(0, 1)` an event affects the day and the day after. A negative `lower_window` reaches back before the event. The helper below fits the model and reads the effect on chosen days.

In [ ]:
def event_effects(holidays: pd.DataFrame, days: list[str], **prophet_kwargs) -> pd.DataFrame:
    """Fit with ``holidays`` and return the playoff and Super Bowl effect on ``days``."""
    r = forecast(df, 365, freq="D", holidays=holidays, **prophet_kwargs)
    return r.predictions.set_index("ds").loc[pd.to_datetime(days), ["playoff", "superbowl"]]


playoff_days = ["2016-01-22", "2016-01-23", "2016-01-24", "2016-01-25", "2016-01-26"]
windows = [(0, 1), (-1, 1), (-2, 3)]
pd.DataFrame(
    {f"window {w}": event_effects(playoffs_and_superbowls(*w), playoff_days)["playoff"] for w in windows}
).round(2)

The playoff game is on 24 January 2016. Wider windows spread the effect over more days, and each day in the window gets its own coefficient, so a long window costs more parameters and risks fitting noise.

**Prior scale.** `holidays_prior_scale` (default 10) sets how freely event effects may be large. A smaller value shrinks them towards zero, most for events seen only a few times. The Super Bowl is on 7 February 2016, and the history holds only two earlier ones (2010 and 2014).

In [ ]:
superbowl_days = ["2016-02-06", "2016-02-07", "2016-02-08"]
pd.DataFrame(
    {f"prior scale {s}": event_effects(holidays, superbowl_days, holidays_prior_scale=s)["superbowl"] for s in [10, 1, 0.05]}
).round(2)

Takeaway: the tightest prior shrinks the Super Bowl effect from about 1.2 to about 1.0, while 10 and 1 are almost the same. The pull is mild here because the spikes are large. It matters more for rare or noisy events, so lower the prior scale when you do not trust an estimate from a few occurrences.

## 8. Additive or multiplicative seasonality

The default is additive: the seasonal swing is the same size however high the trend is. When the swing grows with the level, as in airline passengers, use `seasonality_mode="multiplicative"`, so the seasonal effect is a percentage of the trend. To compare them fairly, hold out the last 24 months, fit each mode on the rest and check the forecasts against the held-out values.

In [ ]:
air = load_example("air_passengers")
train, test = air.iloc[:-24], air.iloc[-24:]
by_mode = {mode: forecast(train, 24, freq="MS", seasonality_mode=mode) for mode in ["additive", "multiplicative"]}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True, layout="constrained")
for ax, (mode, r) in zip(axes, by_mode.items()):
    r.plot(ax=ax, title=mode, subtitle=False, annotate=False)
    ax.scatter(test["ds"], test["y"], s=14, color="black", zorder=6)

In [ ]:
def holdout_rmse(r, actual: pd.DataFrame) -> float:
    merged = r.future.merge(actual, on="ds")
    return float(np.sqrt(((merged["yhat"] - merged["y"]) ** 2).mean()))


pd.Series({mode: holdout_rmse(r, test) for mode, r in by_mode.items()}, name="holdout_rmse").round(1).to_frame()

Black dots are the 24 held-out months. Takeaway: the additive model draws the same-sized seasonal swing at the start and the end, so it undershoots the later, taller peaks; the multiplicative model scales the swing with the trend and misses by less. This is one series and one split, so treat it as an illustration and not a rule.

The components plot shows multiplicative effects as percentages of the trend.

In [ ]:
fig = by_mode["multiplicative"].plot_components()

**Per-component mode.** The mode can be set for each seasonality, holiday group or regressor. Here the model's default stays additive and the yearly seasonality is overridden with `mode="multiplicative"` through `add_seasonality`, which replaces the built-in yearly one. `holidays_mode` and the `mode` argument of `add_regressor` work the same way.

In [ ]:
m_mixed = Prophet(seasonality_mode="additive")
m_mixed.add_seasonality(name="yearly", period=365.25, fourier_order=10, mode="multiplicative")
f_mixed = fit_and_predict(m_mixed, train, periods=24, freq="MS")
print(m_mixed.component_modes)

gap = np.abs(f_mixed["yhat"].tail(24).to_numpy() - by_mode["multiplicative"].future["yhat"].to_numpy()).max()
print(f"Largest difference from seasonality_mode='multiplicative': {gap:.4f}")

Limits: this series has only one seasonality, so the per-component model matches the fully multiplicative one. Mixing modes pays off when components behave differently, for example a weekly pattern of fixed size on top of a yearly one that grows with the level. Prophet cannot tell you which is right; the held-out comparison above can.

## Next

Seasonality and holidays are patterns known from the calendar. Notebook [04. Regressors and weather](04_regressors_and_weather.ipynb) adds outside variables, such as temperature, that explain what the calendar cannot.